# K-means, step by step

K-means splits data into `k` groups. It starts with `k` guesses for the group
centres (the **centroids**) and then repeats two moves until nothing changes:

1. **Assign**: every point joins its nearest centroid.
2. **Update**: every centroid moves to the average of its points.

`glasslearn.cluster.KMeans` works exactly like scikit-learn's `KMeans` (same
parameters, same results), but it also records each of those moves so you can
watch them.

In [ ]:
import glasslearn as gl
from glasslearn.cluster import KMeans

## Some data

Four round blobs of points. `as_frame=True` gives a pandas DataFrame, so the
column names show up in the explanations.

In [ ]:
X = gl.datasets.make_blobs(n_samples=300, centers=4, random_state=0, as_frame=True)
X.head()

## Fit, then explain

Drag the slider or press play. Each step shows what happened and *why*.
Switch between **beginner**, **detailed** and **math** at the top right. The
small curve under the plot is the inertia: the total squared distance from
points to their centroids, which k-means makes as small as it can.

In [ ]:
km = KMeans(n_clusters=4, random_state=0).fit(X)
km.explain()

The fitted model has the usual scikit-learn attributes. `summary()` shows the
final centroids with your column names.

In [ ]:
print(f"{km.n_iter_} iterations, inertia {km.inertia_:.1f}")
km.summary()

## The recording is plain data

Everything the view shows comes from `km.trace_`, one entry per half-step.

In [ ]:
km.trace_.to_frame().head(8)

In [ ]:
step = km.trace_[2]
print(step)
for event in step.events[:2]:
    print(event)

## Experiment: a bad start

Put one starting centroid far away from all the data. After the first
assignment it has no points at all. Watch what k-means does about it, and
notice the warning box.

In [ ]:
import numpy as np

start = np.array([[0.0, 0.0], [1.0, 4.0], [-1.0, 8.0], [30.0, 30.0]])
bad = KMeans(n_clusters=4, init=start, n_init=1).fit(X)
bad.explain(step=2, level="detailed")

## Experiment: features on different scales

K-means measures plain distances, so a feature measured in big numbers
drowns out the others. Here `price` is in dollars and `rooms` is a small count.

In [ ]:
import pandas as pd

rng = np.random.RandomState(5)
houses = pd.DataFrame(
    {"rooms": rng.normal(3, 1, 200), "price": rng.normal(250_000, 80_000, 200)}
)
KMeans(n_clusters=3, random_state=0).fit(houses).explain()

Scaling each column to mean 0 and standard deviation 1 fixes it:

In [ ]:
scaled = (houses - houses.mean()) / houses.std()
KMeans(n_clusters=3, random_state=0).fit(scaled).explain()

## More than two features

With more than two features the plot uses PCA: the two directions in which the
data is most spread out. The axis titles say how much of the spread is kept.
You can also pick two features yourself.

In [ ]:
X5, _ = gl.datasets.make_blobs(n_samples=500, n_features=5, centers=3, random_state=2)
km5 = KMeans(n_clusters=3, random_state=0).fit(X5)
km5.explain()

In [ ]:
km5.explain(projection=("x0", "x3"))

## Sharing a run

`static=True` gives a plain plotly figure with a slider (handy where widgets
do not work), and `save()` writes a standalone HTML file you can open
anywhere, even offline.

In [ ]:
import pathlib
import tempfile

path = km.explain().save(pathlib.Path(tempfile.gettempdir()) / "kmeans_run.html")
print("saved", path)
km.explain(static=True)

## When you do not need the recording

`fit(X, record=False)` gives the identical model, a little faster, with an
empty trace.

In [ ]:
quick = KMeans(n_clusters=4, random_state=0).fit(X, record=False)
print(np.allclose(quick.cluster_centers_, km.cluster_centers_), len(quick.trace_))